In [9]:
# ===============================================
# STEP 1 : Load tokenized data (Pickle) & split
# ===============================================
import pickle, random
from collections import Counter

# ---- Load sentences ----
with open("tokenized_data_complete.pkl", "rb") as f:   # <-- upload your pickle here
    sentences = pickle.load(f)

print("Total sentences:", len(sentences))
print("Example:", sentences[0][:10])

# ---- Shuffle & split ----
random.shuffle(sentences)
val_set  = sentences[:1000]
test_set = sentences[1000:2000]
train_set= sentences[2000:]

print(f"Validation: {len(val_set)} | Test: {len(test_set)} | Train: {len(train_set)}")

# ===============================================
# STEP 2 : Build n-gram counts (1 to 4)
# ===============================================
from itertools import islice

def ngram_counts(corpus, n):
    cnt = Counter()
    for sent in corpus:
        tokens = ["<s>"]*(n-1) + sent + ["</s>"]
        for i in range(len(tokens)-n+1):
            cnt[tuple(tokens[i:i+n])] += 1
    return cnt

uni_counts  = ngram_counts(train_set,1)
bi_counts   = ngram_counts(train_set,2)
tri_counts  = ngram_counts(train_set,3)
quad_counts = ngram_counts(train_set,4)

print("Top unigrams:", uni_counts.most_common(10))


Total sentences: 50001


KeyError: slice(None, 10, None)

In [13]:
# ===============================================
# STEP 1 : Load tokenized data (Pickle) & split
# ===============================================
import pickle, random
from collections import Counter

# ---- Load sentences ----
with open("tokenized_data_complete.pkl", "rb") as f:   # <-- upload your pickle here
    sentences = pickle.load(f)

print("Total sentences:", len(sentences))
print("Example:", sentences[0])

# ---- Shuffle & split ----
random.shuffle(sentences)
val_set  = sentences[:1000]
test_set = sentences[1000:2000]
train_set= sentences[2000:]

print(f"Validation: {len(val_set)} | Test: {len(test_set)} | Train: {len(train_set)}")

# ===============================================
# STEP 2 : Build n-gram counts (1 to 4)
# ===============================================
from itertools import islice

def ngram_counts(corpus, n):
    cnt = Counter()
    for sent_dict in corpus:
        # Access the list of tokens from the dictionary
        tokens = ["<s>"]*(n-1) + sent_dict['sentences'][0]['tokens'] + ["</s>"]
        for i in range(len(tokens)-n+1):
            cnt[tuple(tokens[i:i+n])] += 1
    return cnt

uni_counts  = ngram_counts(train_set,1)
bi_counts   = ngram_counts(train_set,2)
tri_counts  = ngram_counts(train_set,3)
quad_counts = ngram_counts(train_set,4)

print("Top unigrams:", uni_counts.most_common(10))
print("Top bigrams:", bi_counts.most_common(10))
print("Top trigrams:", tri_counts.most_common(10))
print("Top quadgrams:", quad_counts.most_common(10))

Total sentences: 50001
Example: {'document_id': 0, 'original_text': 'लोगों को बिलों संबंधी सुविधा देना ही उनका काम', 'sentences': [{'text': 'लोगों को बिलों संबंधी सुविधा देना ही उनका काम', 'tokens': ['लोगों', 'को', 'बिलों', 'संबंधी', 'सुविधा', 'देना', 'ही', 'उनका', 'काम'], 'word_count': 9}], 'document_stats': {'sentence_count': 1, 'word_count': 9, 'character_count': 45}}
Validation: 1000 | Test: 1000 | Train: 48001
Top unigrams: [(('</s>',), 48001), (('के',), 37809), (('में',), 28753), (('की',), 22100), ((',',), 19105), (('को',), 16198), (('से',), 15046), (('ने',), 13989), (('का',), 11815), (('है',), 10763)]
Top bigrams: [(('.', '</s>'), 9369), (('है।', '</s>'), 8021), (('के', 'लिए'), 4919), (('हैं।', '</s>'), 2846), (('है', '.'), 2453), (('है', 'कि'), 2318), (('कहा', 'कि'), 2044), (('के', 'साथ'), 2012), (('के', 'बाद'), 1879), (('ने', 'कहा'), 1874)]
Top trigrams: [(('है', '.', '</s>'), 2453), (('ने', 'कहा', 'कि'), 1133), (('<s>', '<s>', 'इस'), 952), (('हैं', '.', '</s>'), 936), (('<s>'

In [20]:
# ===============================================
# Good-Turing smoothing for n-grams
# ===============================================
from collections import Counter
import math
import pandas as pd

def good_turing_adjust(counts):
    Nc = Counter(counts.values())          # frequency-of-frequency
    max_c = max(Nc)
    c_star = {}
    for c in range(max_c+1):
        if Nc[c]>0 and Nc.get(c+1,0)>0:
            c_star[c] = (c+1) * Nc[c+1] / Nc[c]
        else:
            c_star[c] = c
    N = sum(counts.values())
    return c_star, Nc, N

def prob_good_turing(ngram, counts, c_star, Nc, N, vocab_size):
    c = counts.get(ngram, 0)
    if c>0:
        return c_star.get(c, c) / N
    else:
        N1 = Nc.get(1, 0)
        return (N1/N) / (vocab_size - len(counts))

# ---- prepare smoothing for trigram as example ----
vocab = set(tok for s in train_set for tok in s['sentences'][0]['tokens']) # Also fix the vocab calculation
tri_cstar, tri_Nc, tri_N = good_turing_adjust(tri_counts)

# ===============================================
# Sentence probability under Good-Turing
# ===============================================
def sentence_prob(sent_dict, n, counts, c_star, Nc, N, vocab_size):
    # Access the list of tokens from the dictionary
    tokens = ["<s>"]*(n-1) + sent_dict['sentences'][0]['tokens'] + ["</s>"]
    logp = 0.0
    for i in range(len(tokens)-n+1):
        ng = tuple(tokens[i:i+n])
        p = prob_good_turing(ng, counts, c_star, Nc, N, vocab_size)
        logp += math.log(p if p>0 else 1e-12)
    return math.exp(logp)

for s in val_set[:5]:
    print("Sentence prob:", sentence_prob(s,3,tri_counts,tri_cstar,tri_Nc,tri_N,len(vocab)))

# ===============================================
# Build table for C, Nc, C* (top 100)
# ===============================================
rows=[]
for c in sorted(tri_Nc.keys())[:100]:
    rows.append([c, tri_Nc[c], round(tri_cstar.get(c, c), 3)])
df = pd.DataFrame(rows, columns=["C (MLE)", "Nc", "C*"])
print(df.head(101))

Sentence prob: 2.2468154262780678e-130
Sentence prob: 1.6131132996498106e-232
Sentence prob: 1.0000000000000094e-120
Sentence prob: 2.551418986296671e-156
Sentence prob: 4.5176044e-317
    C (MLE)      Nc       C*
0         1  624701    0.128
1         2   39838    0.933
2         3   12395    1.854
3         4    5746    2.817
4         5    3237    3.666
..      ...     ...      ...
95       98       2   49.500
96       99       1  300.000
97      100       3   33.667
98      101       1  306.000
99      102       3   34.333

[100 rows x 3 columns]


In [19]:
# ===============================================
# Deleted Interpolated Smoothing for Quadrigrams
# ===============================================
import numpy as np
from tqdm import tqdm   # <-- NEW

counts_all = [uni_counts, bi_counts, tri_counts, quad_counts]

def deleted_interp_prob(sent_dict, lambdas, counts_all):
    n = 4
    tokens = ["<s>"]*(n-1) + sent_dict['sentences'][0]['tokens'] + ["</s>"]
    logp = 0.0
    for i in range(len(tokens)-3):
        p = 0.0
        for k in range(4):
            hist  = tuple(tokens[i:i+k]) if k>0 else ()
            ngram = tuple(tokens[i:i+k+1])
            denom = counts_all[k-1].get(hist, sum(counts_all[k-1].values())) if k>0 else sum(counts_all[k].values())
            num   = counts_all[k].get(ngram, 0)
            pk = num/denom if denom>0 else 0
            p += lambdas[k]*pk
        logp += np.log(p if p>0 else 1e-12)
    return np.exp(logp)

# ---- Grid search for best lambdas ----
grid = np.linspace(0, 1, 5)  # step = 0.25
best = None
best_score = -1e9
sample_val = val_set[:200]

# total combinations for tqdm
total = sum(1 for l1 in grid for l2 in grid for l3 in grid if 1 - l1 - l2 - l3 >= 0)

with tqdm(total=total, desc="Searching λ") as pbar:
    for l1 in grid:
        for l2 in grid:
            for l3 in grid:
                l4 = 1 - l1 - l2 - l3
                if l4 < 0:
                    continue
                lamb = [l1, l2, l3, l4]
                score = sum(np.log(deleted_interp_prob(s, lamb, counts_all) + 1e-12)
                            for s in sample_val)
                if score > best_score:
                    best_score, best = score, lamb
                pbar.update(1)

print("Best λ:", best)
print("Perplexity:", np.exp(-best_score / len(sample_val)))


Searching λ: 100%|██████████| 35/35 [37:21<00:00, 64.04s/it]

Best λ: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0)]
Perplexity: 465531416168.3252


In [21]:
print("Best lambda values:", best)

Best lambda values: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0)]
